# Tutorial 01 — analytic X-cycle, DX_pol, and DPm

This notebook adapts ideas from `dummy_Xcycle.ipynb` to the current `pyna` API.

Conventions used here:
- `X_pol = (R, Z)` is the cylindrical poloidal orbit coordinate.
- `DX_pol` is the variational matrix along the continuous-time field-line flow.
- `DPm` is the full-period Jacobian of the Poincare map `P^m`.
- local outputs are written to `D:/MCFdata/tutorials/analytic_Xcycle/`.


In [ ]:
from pathlib import Path
import json
import numpy as np
from scipy.integrate import solve_ivp

from pyna.topo.cycle import find_cycle
from pyna.topo.monodromy import compute_DPm_on_cycle, orbit_shift_under_perturbation

OUTDIR = Path(r'D:/MCFdata/tutorials/analytic_Xcycle')
OUTDIR.mkdir(parents=True, exist_ok=True)
OUTDIR


In [ ]:
# Analytic dummy X-cycle model adapted from MCF_scripts/dummy_Xcycle.ipynb
Rax, Zax = 1.0, 0.0
Rell, Zell = 0.3, 0.5
phi0 = 0.0
BPhiax = 2.5
m_turns = 3
n_mode = 1
iota = n_mode / m_turns
dTETdphi = 1.0 / 3.0
lambda_u = np.exp(1/5)
lambda_s = np.exp(-1/5)

def cycle_RZ(phi):
    return np.array([
        Rell * np.cos(iota * phi + phi0) + Rax,
        Zell * np.sin(iota * phi + phi0) + Zax,
    ])

def theta_u(phi):
    return phi / 3 + np.pi / 2 + np.pi / 9

def theta_s(phi):
    return phi / 3 + np.pi / 2 - np.pi / 9

def Bphi(phi, X_pol):
    return Rax * BPhiax / X_pol[0]

def BR0_BZ0(phi):
    Rc = cycle_RZ(phi)[0]
    BR0 = -iota * Rell * np.sin(iota * phi + phi0) / Rc * Bphi(phi, cycle_RZ(phi))
    BZ0 = +iota * Zell * np.cos(iota * phi + phi0) / Rc * Bphi(phi, cycle_RZ(phi))
    return np.array([BR0, BZ0])

def A_dummy(phi):
    V = np.array([[np.cos(theta_u(phi)), np.cos(theta_s(phi))],
                  [np.sin(theta_u(phi)), np.sin(theta_s(phi))]])
    Lam = np.array([[np.log(abs(lambda_u)) / (2 * m_turns * np.pi), 0.0],
                    [0.0, np.log(abs(lambda_s)) / (2 * m_turns * np.pi)]])
    return V @ Lam @ np.linalg.inv(V) + np.array([[0.0, -dTETdphi], [dTETdphi, 0.0]])

def BR_BZ(phi, X_pol):
    Rc = cycle_RZ(phi)[0]
    return BR0_BZ0(phi) + Bphi(phi, cycle_RZ(phi)) / Rc * (A_dummy(phi) @ (X_pol - cycle_RZ(phi)))

def field_func(rzphi):
    r, z, phi = map(float, rzphi)
    X_pol = np.array([r, z])
    BR, BZ = BR_BZ(phi, X_pol)
    BP = Bphi(phi, X_pol) / r
    return np.array([BR, BZ, BP])

seed = np.array([cycle_RZ(0.0)[0] + 5e-3, cycle_RZ(0.0)[1] - 5e-3, 0.0])
orbit = find_cycle(field_func, seed, n_turns=m_turns, dt=0.05, max_iter=50, tol=1e-10)
orbit.rzphi0 if orbit is not None else None


In [ ]:
analysis = compute_DPm_on_cycle(field_func, orbit, dt_output=0.05)

phi_e = orbit.rzphi0[2] + 2 * np.pi * orbit.period_m
DPm_from_DX_pol = analysis.DX_pol_arr[-1]
DPm_from_analysis = analysis.DPm

summary = {
    'phi_start': float(orbit.rzphi0[2]),
    'phi_e': float(phi_e),
    'period_m': int(orbit.period_m),
    'DPm': DPm_from_analysis.tolist(),
    'DPm_minus_DX_pol_end_maxabs': float(np.max(np.abs(DPm_from_analysis - DPm_from_DX_pol))),
    'DPm_ring_init_minus_DPm_maxabs': float(np.max(np.abs(analysis.DPm_arr[0] - DPm_from_analysis))),
    'eigvals_DPm': np.linalg.eigvals(DPm_from_analysis).tolist(),
}
(OUTDIR / 'analytic_Xcycle_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary


In [ ]:
# Example perturbation: small radial-localized perturbation with zero delta Bphi
def delta_field_func(rzphi):
    r, z, phi = map(float, rzphi)
    Rc, Zc = cycle_RZ(phi)
    amp = 1e-3 * np.exp(-((r - Rc)**2 + (z - Zc)**2) / 0.02**2)
    return np.array([amp, -0.5 * amp, 0.0])

delta_X_pol = orbit_shift_under_perturbation(field_func, delta_field_func, orbit, analysis)
np.savez(OUTDIR / 'analytic_Xcycle_shift.npz',
         phi=analysis.phi_arr,
         trajectory=analysis.trajectory,
         DX_pol=analysis.DX_pol_arr,
         DPm_ring=analysis.DPm_arr,
         delta_X_pol=delta_X_pol)
delta_X_pol[:3]
